# 09 — Earth & Geoscience Foundation Models for Critical Minerals

This notebook demonstrates how **Earth observation and geoscience foundation models** can
enhance critical-mineral prospectivity analysis beyond what is possible with tabular
production statistics alone.

Foundation models pre-trained on massive geospatial corpora encode terrain, climate,
vegetation, and geological text into rich feature vectors. When combined with BGS production
data, these representations can reveal geographic patterns that purely numeric clustering
misses.

---

## Models Featured

| # | Model | Organisation | Size | HuggingFace / GitHub | Capability |
|---|-------|-------------|------|----------------------|------------|
| 1 | **SatCLIP** | Microsoft | ViT-L/16 | [microsoft/SatCLIP-ViT16-L10](https://huggingface.co/microsoft/SatCLIP-ViT16-L10) · [github](https://github.com/microsoft/satclip) | Location encoder — embeds any (lat, lon) from satellite imagery |
| 2 | **Prithvi-100M** | NASA / IBM | 100 M params | [ibm-nasa-geospatial/Prithvi-100M](https://huggingface.co/ibm-nasa-geospatial/Prithvi-100M) | Geospatial ViT for remote-sensing time-series |
| 3 | **Clay** | Clay Foundation | ViT | [made-with-clay/Clay](https://huggingface.co/made-with-clay/Clay) · [github](https://github.com/Clay-foundation/model) | Self-supervised EO model for satellite imagery patches |
| 4 | **K2 / GeoLLaMA** | Community | 7 B / 65 B | [daven3/k2-7b](https://huggingface.co/daven3/k2-7b) · [GeoGalactica](https://github.com/acl2023-geodiscovery/GeoGalactica) | LLM fine-tuned on geoscience text |

---

## Notebook Outline

1. **Geographic Data Preparation** — country centroids merged with BGS production data
2. **SatCLIP — Location Embeddings** — encode country centroids into 512-D feature vectors
3. **Prithvi — Architecture Overview** — how to integrate for mine-site remote-sensing analysis
4. **Clay Foundation Model — Architecture Overview** — EO patch encoding pathway
5. **Geoscience NLP: K2 / GeoLLaMA** — geological context via a geoscience LLM
6. **Combined Analysis** — location embeddings + production-profile text embeddings
7. **Geographic Visualisation** — world map coloured by combined cluster
8. **Summary & Next Steps**

> **Data source:** British Geological Survey (BGS) World Mineral Statistics  
> **Rows:** ~59 K production/trade records across 60+ commodities  
> **INL note:** For HPC-hosted LLM endpoints, set `LLM_BASE_URL` / `LLM_API_KEY` in section 5.

In [ ]:
# ── Core dependencies (always available) ──────────────────────────────────────
# Model-specific installs are deferred to their own sections so the notebook
# remains usable even when optional packages are absent.

import warnings
warnings.filterwarnings('ignore')

import pathlib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = pathlib.Path("../data/bgs_data")
CSV_PATH = DATA_DIR / "bgs_critical_minerals_production.csv"

# Fallback for notebooks run from the repo root
if not CSV_PATH.exists():
    CSV_PATH = pathlib.Path("data/bgs_data/bgs_critical_minerals_production.csv")

print(f"Data file : {CSV_PATH}")
print(f"Exists    : {CSV_PATH.exists()}")

## 1. Geographic Data Preparation

The BGS dataset records production by **country** (ISO3 code) but does not include
mine-level coordinates.  To feed location encoders such as SatCLIP we map each
country to its approximate geographic centroid — a reasonable proxy for the
country's mineral-producing regions at the national scale.

We hardcode centroids for the ~50 most productive countries in the dataset and then
left-join them onto the aggregated BGS production table so that every row in the
analysis carries `(lat, lon)` alongside production statistics.

> **Upgrade path:** Replace centroid coordinates with mine-level coordinates from
> [USGS MRDS](https://mrdata.usgs.gov/mrds/), [S&P SNL Metals & Mining](https://www.spglobal.com/commodityinsights/en/ci/products/snl-metals-mining.html),
> or [OpenStreetMap](https://wiki.openstreetmap.org/wiki/Tag:landuse%3Dquarry) for
> finer-grained geospatial analysis.

In [ ]:
# ── Country centroid coordinates (approximate, WGS-84) ────────────────────────
COUNTRY_COORDS = {
    "CHN": (35.86, 104.20),  "AUS": (-25.27, 133.77), "USA": (37.09, -95.71),
    "BRA": (-14.24, -51.93), "RUS": (61.52,  105.32),  "IND": (20.59,  78.96),
    "ZAF": (-30.56,  22.94), "CAN": (56.13, -106.35),  "COD": (-4.04,  21.76),
    "CHL": (-35.68, -71.54), "PER": (-9.19,  -75.02),  "MEX": (23.63, -102.55),
    "IDN": (-0.79,  113.92), "ARG": (-38.42, -63.62),  "KAZ": (48.02,  66.92),
    "PHL": (12.88,  121.77), "MYS": (4.21,   101.98),  "THA": (15.87, 100.99),
    "TUR": (38.96,   35.24), "COL": (4.57,   -74.30),  "NGA": (9.08,    8.68),
    "GHA": (7.95,    -1.02), "TZA": (-6.37,   34.89),  "MOZ": (-18.67, 35.53),
    "ZWE": (-19.02,  29.15), "ZMB": (-13.13,  27.85),  "MMR": (21.91,  95.96),
    "VNM": (14.06,  108.28), "PAK": (30.38,   69.35),  "IRN": (32.43,  53.69),
    "SAU": (23.89,   45.08), "UKR": (48.38,   31.17),  "POL": (51.92,  19.15),
    "SWE": (60.13,   18.64), "NOR": (60.47,    8.47),  "FIN": (61.92,  25.75),
    "DEU": (51.17,   10.45), "FRA": (46.23,    2.21),  "GBR": (55.38,  -3.44),
    "JPN": (36.20,  138.25), "KOR": (35.91,  127.77),  "NZL": (-40.90, 174.89),
    "CUB": (21.52,  -77.78), "BOL": (-16.29, -63.59),  "ECU": (-1.83,  -78.18),
    "GIN": (9.95,    -9.70), "SLE": (8.46,   -11.78),  "NAM": (-22.96,  18.49),
    "BWA": (-22.33,  24.68), "MNG": (46.86,  103.85),  "NCL": (-20.90, 165.62),
}

coords_df = pd.DataFrame(
    [(iso3, lat, lon) for iso3, (lat, lon) in COUNTRY_COORDS.items()],
    columns=["country_iso3", "lat", "lon"],
)
print(f"Country centroids loaded: {len(coords_df)} countries")
coords_df.head()

In [ ]:
# ── Load BGS data and aggregate ───────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Raw rows : {len(df_raw):,}")
print(f"Columns  : {list(df_raw.columns)}")

# Keep production records, coerce numerics
df = df_raw.copy()
df = df[df["statistic_type"].str.strip().str.lower() == "production"].copy()
df["year"]     = pd.to_numeric(df["year"],     errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df = df.dropna(subset=["year", "quantity", "country", "commodity"])
df = df[df["quantity"] > 0]

# Restrict to most recent 5-year window
max_year    = int(df["year"].max())
year_cutoff = max_year - 4
df = df[df["year"] >= year_cutoff].copy()

# Aggregate: country x commodity, mean quantity
agg = (
    df.groupby(["country", "country_iso3", "commodity"], as_index=False)["quantity"]
    .mean()
    .rename(columns={"quantity": "mean_qty"})
)

# Country-level summary (total production, top mineral)
country_summary = (
    agg.sort_values("mean_qty", ascending=False)
    .groupby(["country", "country_iso3"], as_index=False)
    .agg(
        total_production=("mean_qty", "sum"),
        num_minerals=("commodity", "nunique"),
        top_mineral=("commodity", "first"),   # first after sort = largest
    )
)

# Merge with centroid coordinates
geo_df = country_summary.merge(coords_df, on="country_iso3", how="inner")
print(f"\nCountries with production + coordinates: {len(geo_df)}")
print(f"Year window: {year_cutoff}–{max_year}")
geo_df.head()

## 2. SatCLIP — Location Embeddings

[**SatCLIP**](https://github.com/microsoft/satclip) is a **contrastive location encoder**
developed by Microsoft Research.  It is trained by aligning satellite imagery patches
(from Sentinel-2 / Landsat) with their geographic coordinates using a CLIP-style objective.
The result is a model that can encode any `(lat, lon)` pair into a high-dimensional vector
capturing:

- **Terrain** — elevation, slope, aspect (inferred from spectral patterns)
- **Vegetation** — biome type, NDVI regime
- **Climate** — arid/humid/polar signatures
- **Land-use** — urban, agricultural, bare rock, mine waste

For critical-mineral analysis these embeddings are valuable because the environmental
setting of a deposit strongly correlates with mineralisation style.  Porphyry copper
deposits cluster in arid Andean or South-West Pacific settings; laterite nickel deposits
in humid tropical zones; pegmatite lithium in continental interiors.

### Available checkpoints

| Checkpoint | Backbone | Harmonics (L) | Notes |
|------------|----------|---------------|-------|
| `satclip-resnet18-l10` | ResNet-18 | 10 | Fastest, smallest |
| `satclip-resnet50-l10` | ResNet-50 | 10 | Balanced |
| `satclip-vit16-l10`    | ViT-B/16  | 10 | Best quality |
| `satclip-vit16-l40`    | ViT-B/16  | 40 | Highest spatial resolution |

### Installation
```bash
pip install git+https://github.com/microsoft/satclip.git
```

In [ ]:
import torch

# ── Country centroid arrays ────────────────────────────────────────────────────
countries  = geo_df["country_iso3"].tolist()
lat_arr    = geo_df["lat"].values
lon_arr    = geo_df["lon"].values
coords_np  = np.column_stack([lat_arr, lon_arr])   # shape (N, 2)

SATCLIP_CHECKPOINT = "satclip-vit16-l10"   # change to resnet18 for faster loading

try:
    from satclip.load import get_satclip_model

    print(f"Loading SatCLIP checkpoint: {SATCLIP_CHECKPOINT} ...")
    satclip_model = get_satclip_model(SATCLIP_CHECKPOINT, device="cpu")
    satclip_model.train(False)   # set to inference mode

    coords_tensor = torch.tensor(coords_np, dtype=torch.float32)

    with torch.no_grad():
        location_embeddings = satclip_model(coords_tensor).numpy()

    print(f"SatCLIP embeddings shape : {location_embeddings.shape}")
    print(f"  rows = {location_embeddings.shape[0]} countries")
    print(f"  cols = {location_embeddings.shape[1]} dimensions")
    SATCLIP_AVAILABLE = True

except (ImportError, Exception) as exc:
    print(f"SatCLIP not available: {exc}")
    print("\nInstall with:")
    print("  pip install git+https://github.com/microsoft/satclip.git")
    print("\nFalling back to handcrafted coordinate feature vector (6-D).")

    # Fallback: encode (lat, lon) with simple geographic features
    lat_r = np.radians(lat_arr)
    lon_r = np.radians(lon_arr)
    location_embeddings = np.column_stack([
        lat_arr,                           # raw latitude
        lon_arr,                           # raw longitude
        np.abs(lat_arr),                   # distance from equator
        np.sin(lat_r) * np.cos(lon_r),    # 3-D unit-sphere x
        np.sin(lat_r) * np.sin(lon_r),    # 3-D unit-sphere y
        np.cos(lat_r),                     # 3-D unit-sphere z
    ]).astype(np.float32)

    print(f"Fallback coordinate features shape: {location_embeddings.shape}")
    SATCLIP_AVAILABLE = False

### 2a. Cluster Countries by Location Embedding

We project the location embeddings to 2-D with **UMAP** and cluster with **HDBSCAN**.
Countries sharing similar environmental signatures (e.g. Andean producers, arid African
producers, boreal producers) should emerge as coherent clusters.

This is complementary to the production-profile clustering in notebook 04:
- Notebook 04 clusters by **what** countries produce.
- This section clusters by **where** they are — the environmental and geological setting.

In [ ]:
import umap
import hdbscan
from sklearn.preprocessing import normalize

# L2-normalise so cosine distance == Euclidean in HDBSCAN
loc_emb_norm = normalize(location_embeddings, norm="l2")

# ── UMAP projection (for visualisation only) ───────────────────────────────────
print("Running UMAP on location embeddings ...")
reducer_loc = umap.UMAP(
    n_components=2,
    n_neighbors=min(10, len(loc_emb_norm) - 1),
    min_dist=0.1,
    metric="cosine",
    random_state=42,
)
umap_loc = reducer_loc.fit_transform(loc_emb_norm)
print(f"UMAP output shape: {umap_loc.shape}")

# ── HDBSCAN clustering in full embedding space ─────────────────────────────────
print("Running HDBSCAN ...")
clusterer_loc = hdbscan.HDBSCAN(
    min_cluster_size=3,
    min_samples=2,
    metric="euclidean",
    cluster_selection_method="eom",
)
loc_labels = clusterer_loc.fit_predict(loc_emb_norm)

n_clusters_loc = len(set(loc_labels)) - (1 if -1 in loc_labels else 0)
n_noise_loc    = int((loc_labels == -1).sum())
print(f"Location clusters: {n_clusters_loc}  |  Noise: {n_noise_loc}")

# ── Assemble result ────────────────────────────────────────────────────────────
loc_result_df = geo_df.copy()
loc_result_df["umap_x"]        = umap_loc[:, 0]
loc_result_df["umap_y"]        = umap_loc[:, 1]
loc_result_df["loc_cluster"]   = loc_labels
loc_result_df["cluster_label"] = loc_result_df["loc_cluster"].apply(
    lambda c: f"Geo-Cluster {c}" if c >= 0 else "Noise"
)

loc_result_df[["country", "cluster_label", "top_mineral", "lat", "lon"]].head(10)

In [ ]:
# ── UMAP scatter coloured by location cluster ──────────────────────────────────
palette = pc.qualitative.Plotly + pc.qualitative.D3 + pc.qualitative.G10
unique_clusters = sorted(loc_result_df["cluster_label"].unique())
cluster_only    = [l for l in unique_clusters if l != "Noise"]
colour_map      = {l: palette[i % len(palette)] for i, l in enumerate(cluster_only)}
colour_map["Noise"] = "#cccccc"

MAX_B, MIN_B = 50, 6
log_prod = np.log1p(loc_result_df["total_production"])
loc_result_df["bubble_size"] = (
    (log_prod - log_prod.min()) / (log_prod.max() - log_prod.min() + 1e-9)
    * (MAX_B - MIN_B) + MIN_B
)

emb_label = "SatCLIP Location Embeddings" if SATCLIP_AVAILABLE else "Coordinate Features (SatCLIP fallback)"

fig_umap_loc = px.scatter(
    loc_result_df,
    x="umap_x",
    y="umap_y",
    color="cluster_label",
    color_discrete_map=colour_map,
    size="bubble_size",
    size_max=MAX_B,
    hover_name="country",
    hover_data={
        "top_mineral":      True,
        "lat":              ":.2f",
        "lon":              ":.2f",
        "total_production": ":.2e",
        "cluster_label":    False,
        "umap_x":           False,
        "umap_y":           False,
        "bubble_size":      False,
    },
    title=f"Country Clusters by {emb_label} (UMAP + HDBSCAN)",
    labels={"cluster_label": "Cluster", "umap_x": "UMAP-1", "umap_y": "UMAP-2"},
    template="plotly_white",
    width=950,
    height=620,
)
fig_umap_loc.update_layout(
    legend=dict(title="Cluster", itemsizing="constant"), font_size=13
)
fig_umap_loc.show()

## 3. Prithvi — Geospatial Foundation Model

[**Prithvi-100M**](https://huggingface.co/ibm-nasa-geospatial/Prithvi-100M) is a masked
autoencoder (MAE) developed by **NASA** and **IBM** and trained on 4 million Harmonized
Landsat Sentinel-2 (HLS) tiles covering the contiguous US.  Its architecture is a plain
Vision Transformer (ViT-L) that accepts **6-band multispectral time-series** input
(Blue, Green, Red, Narrow-NIR, SWIR-1, SWIR-2) with a temporal depth of 3 frames.

### Why Prithvi matters for critical minerals

| Application | Satellite bands used | Mineral relevance |
|-------------|---------------------|-------------------|
| Hydrothermal alteration mapping | SWIR-1, SWIR-2 | Phyllosilicate minerals indicate porphyry / epithermal environments |
| Iron-oxide mapping | Red, NIR | Gossans above sulphide ore bodies |
| Vegetation stress (geobotany) | Red-Edge, NIR | Anomalous elemental concentrations affect canopy spectral response |
| Land-use change at mine sites | All bands, temporal | Monitor extraction rates and tailings expansion |
| Flood risk around tailings | Temporal NIR/SWIR | Environmental hazard screening |

### Integration pathway
1. Download HLS tiles (free) from [NASA EarthData](https://earthdata.nasa.gov/) for your AOI.
2. Extract 224x224 px chips centred on known deposit coordinates.
3. Stack 3 temporal acquisitions to shape `(B, T, C, H, W)` = `(batch, 3, 6, 224, 224)`.
4. Run through `Prithvi-100M` encoder to get 768-D patch-level features.
5. Pool features to per-deposit embeddings and feed into a downstream classifier.

In [ ]:
# ── Prithvi-100M: architecture inspection ─────────────────────────────────────
# We inspect the config without downloading the full weights (~400 MB) to keep
# the notebook fast.  Set LOAD_PRITHVI_WEIGHTS = True to also download the model.
LOAD_PRITHVI_WEIGHTS = False

try:
    from transformers import AutoConfig, AutoModel

    print("Fetching Prithvi-100M config from HuggingFace Hub ...")
    prithvi_config = AutoConfig.from_pretrained(
        "ibm-nasa-geospatial/Prithvi-100M",
        trust_remote_code=True,
    )

    print("\nPrithvi-100M Configuration:")
    for attr in [
        "hidden_size", "num_hidden_layers", "num_attention_heads",
        "intermediate_size", "image_size", "patch_size",
        "num_frames", "num_channels", "tubelet_size",
    ]:
        val = getattr(prithvi_config, attr, "N/A")
        print(f"  {attr:<28}: {val}")

    if LOAD_PRITHVI_WEIGHTS:
        print("\nLoading model weights (~400 MB) ...")
        prithvi_model = AutoModel.from_pretrained(
            "ibm-nasa-geospatial/Prithvi-100M",
            trust_remote_code=True,
        )
        prithvi_model.train(False)   # set to inference mode
        n_params = sum(p.numel() for p in prithvi_model.parameters()) / 1e6
        print(f"  Parameters loaded: {n_params:.1f} M")

except Exception as exc:
    print(f"Could not fetch Prithvi config: {exc}")
    print("\nInstall requirements:")
    print("  pip install transformers torch")
    print("\nData requirements (for full inference):")
    print("  HLS Landsat/Sentinel-2 tiles — NASA EarthData (free account required)")
    print("  Input shape: (batch, time=3, channels=6, height=224, width=224)")
    print("  Bands: Blue, Green, Red, Narrow-NIR, SWIR-1, SWIR-2")

print("\nPrithvi applications for critical mineral analysis:")
applications = [
    ("Hydrothermal alteration mapping",
     "SWIR bands discriminate phyllosilicates — porphyry/epithermal indicators"),
    ("Iron-oxide / gossan detection",
     "Red + NIR highlight gossans overlying sulphide ore bodies"),
    ("Geobotanical anomaly detection",
     "Metal-stressed vegetation shows distinctive NIR response"),
    ("Mine-site change detection",
     "Temporal stacking reveals pit expansion, tailings growth"),
    ("Tailings flood risk",
     "NIR/SWIR flood mapping around mine waste facilities"),
]
for app, detail in applications:
    print(f"  [{app}]")
    print(f"    {detail}")

## 4. Clay Foundation Model

[**Clay**](https://github.com/Clay-foundation/model) is an open Earth observation foundation
model released under the MIT licence by the [Clay Foundation](https://made-with-clay.com/).
It is a **masked-patch self-supervised ViT** trained on ~70 TB of globally sampled
Sentinel-2, Sentinel-1 SAR, and DEM data.

### Key differences from Prithvi

| Feature | Prithvi-100M | Clay |
|---------|-------------|------|
| Training data | HLS (Landsat + S-2, CONUS) | Sentinel-1 + Sentinel-2 + DEM (global) |
| Modalities | Optical only | Optical + SAR + elevation |
| Temporal depth | 3 frames | Variable (timestamped embeddings) |
| Architecture | Plain ViT MAE | ViT with metadata-conditioned patch embedding |
| Use case focus | Change detection, segmentation | Any-time, any-place EO encoding |

### Clay's metadata embedding
Clay accepts **timestamped, georeferenced** patches and embeds metadata (date, lat/lon,
sensor, cloud cover) as additional tokens — making it uniquely suited to integrating
temporal production data with EO observations.

### Installation
```bash
pip install clay-foundation  # or clone from github.com/Clay-foundation/model
```

In [ ]:
# ── Clay Foundation Model: architecture overview ───────────────────────────────
try:
    from transformers import AutoConfig

    print("Fetching Clay config from HuggingFace Hub ...")
    clay_config = AutoConfig.from_pretrained(
        "made-with-clay/Clay",
        trust_remote_code=True,
    )
    print("\nClay Model Configuration:")
    for attr in [
        "hidden_size", "num_hidden_layers", "num_attention_heads",
        "patch_size", "image_size", "num_channels",
    ]:
        val = getattr(clay_config, attr, "N/A")
        print(f"  {attr:<28}: {val}")

except Exception as exc:
    print(f"Clay config not available via AutoConfig: {exc}")
    print("Clay uses a custom checkpoint format — load directly:")
    print("  from huggingface_hub import hf_hub_download")
    print("  ckpt = hf_hub_download('made-with-clay/Clay', 'clay-v1-base.ckpt')")

print("\nClay integration pathway for mineral prospectivity:")
steps = [
    "1. Download Sentinel-2 L2A tiles for deposit/prospect AOIs via Copernicus Open Access Hub.",
    "2. Optionally add Sentinel-1 SAR (for lithology under cloud cover / vegetation).",
    "3. Add SRTM/Copernicus DEM as elevation channel.",
    "4. Package into 256x256 px chips with acquisition date + lat/lon centroid.",
    "5. Run through Clay encoder to get 768-D or 1024-D patch CLS token.",
    "6. Fine-tune a linear head on labelled deposit/non-deposit chips.",
    "7. Combine Clay EO embeddings with SatCLIP location embeddings + BGS production features.",
]
for s in steps:
    print(f"  {s}")

print("\nFree satellite data sources:")
print("  Sentinel-2 : Copernicus Open Access Hub — https://scihub.copernicus.eu")
print("  Landsat    : USGS Earth Explorer       — https://earthexplorer.usgs.gov")
print("  HLS        : NASA EarthData            — https://earthdata.nasa.gov")
print("  SRTM DEM   : NASA LP DAAC              — https://lpdaac.usgs.gov")

## 5. Geoscience NLP: K2 / GeoLLaMA

**K2** (also called GeoLLaMA) is a LLaMA-based language model fine-tuned on a large corpus
of geoscience literature, including:

- Peer-reviewed journal articles (Journal of Geophysical Research, Geology, Economic Geology, etc.)
- USGS open-file reports and mineral resource assessments
- GSA bulletins, PhD theses in geoscience

Available sizes: `daven3/k2-7b` (7 B parameters, fits on a single A100 GPU) and
`daven3/k2-65b` (65 B parameters, requires multi-GPU or quantisation).

For resource-constrained environments, [GeoGalactica](https://github.com/acl2023-geodiscovery/GeoGalactica)
(30 B, Falcon backbone) offers similar geoscience domain knowledge.

### Use cases for critical minerals

| Task | Prompt style | Output |
|------|-------------|--------|
| Deposit type description | "Describe the geological setting of Kiruna-type iron-oxide deposits" | Narrative geological context |
| Indicator minerals | "What pathfinder elements indicate lithium pegmatite mineralisation?" | Geochemical target list |
| Risk assessment text | "Summarise the geological risks for the Atacama lithium brine deposits" | Risk narrative |
| Literature Q&A | "According to geoscience literature, where are REE skarn deposits found?" | Grounded answer |

### INL HPC Integration

If you have access to an INL HPC endpoint running a vLLM or Ollama server, you can call
any geoscience model (K2, GeoGalactica, or a general model like Llama-3) via the OpenAI
API-compatible interface.  Set `LLM_BASE_URL` and `LLM_API_KEY` below.

In [ ]:
# ── Geoscience LLM via OpenAI-compatible endpoint ─────────────────────────────
# Configure these to point to your INL HPC vLLM / Ollama instance.
# Leave as None to use a graceful fallback (static reference answers).
LLM_BASE_URL = None   # e.g. "https://your-hpc-endpoint.inl.gov/v1"
LLM_API_KEY  = None   # e.g. "Bearer <token>"
LLM_MODEL    = "daven3/k2-7b"   # or whatever model is served at your endpoint

MINERAL_QUERIES = [
    "What geological formations and tectonic settings are associated with lithium brine deposits?",
    "Describe the typical host rocks and pathfinder elements for rare earth element (REE) deposits.",
    "What are the key geochemical indicators of cobalt mineralisation in laterite and sulphide settings?",
]

# ── Static fallback answers (used when endpoint is not configured) ─────────────
STATIC_ANSWERS = [
    (
        "Lithium brine deposits are hosted in closed-basin salt lakes (salares) at elevations "
        "above 3,500 m in the Central Andes (Chile, Bolivia, Argentina). The brines are "
        "sourced from leaching of volcanic arc rocks and concentrated by extreme aridity and "
        "solar evaporation. Key geological controls: Neogene extensional faulting, volcanism "
        "providing a lithium-rich source, and hydrologically closed basins with minimal "
        "freshwater dilution. The Puna/Altiplano plateau setting — Miocene to Holocene "
        "ignimbrite packages — is the archetype."
    ),
    (
        "REE deposits occur in multiple geological settings: (1) Carbonatites — alkaline "
        "magmatic intrusions rich in LREE (Bayan Obo, China; Mountain Pass, USA); (2) "
        "Ion-adsorption clays — deeply weathered granites in subtropical southern China, "
        "hosting HREE adsorbed on clay minerals; (3) Monazite-xenotime-bearing placers in "
        "beach and fluvial sands; (4) Peralkaline granites and syenites (Thor Lake, Canada). "
        "Pathfinder elements: Ba, Sr, Nb, Ta, P, F. Host rocks include carbonatites, "
        "nepheline syenites, and granitic pegmatites."
    ),
    (
        "Cobalt mineralisation occurs in two main settings: (1) Laterite profiles — Co "
        "enriches in limonite horizons above ultramafic rocks (Ni-Co laterites, Philippines, "
        "New Caledonia); geochemical indicators include Ni:Co ratios <10 and elevated Mn, Fe. "
        "(2) Sediment-hosted Cu-Co deposits (DRC Copperbelt) — stratabound in Neoproterozoic "
        "meta-sediments; pathfinders are Cu, Mo, U, and organic-carbon-rich marker horizons. "
        "Magmatic Ni-Cu-PGE sulphide settings (Sudbury, Norilsk) are a third Co source, "
        "with pyrite-pyrrhotite-pentlandite as indicator mineralogy."
    ),
]

if LLM_BASE_URL and LLM_API_KEY:
    try:
        from openai import OpenAI
        client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
        print(f"Using LLM endpoint : {LLM_BASE_URL}")
        print(f"Model              : {LLM_MODEL}\n")

        for query in MINERAL_QUERIES:
            print(f"Q: {query}")
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": query}],
                max_tokens=350,
                temperature=0.2,
            )
            print(f"A: {resp.choices[0].message.content.strip()}\n")
            print("-" * 70)

    except Exception as exc:
        print(f"LLM endpoint error: {exc}")
        print("Falling back to static reference answers.\n")
        for query, answer in zip(MINERAL_QUERIES, STATIC_ANSWERS):
            print(f"Q: {query}")
            print(f"A (static): {answer}\n")
            print("-" * 70)
else:
    print("LLM_BASE_URL not configured — showing static geoscience reference answers.")
    print("Set LLM_BASE_URL and LLM_API_KEY above to query a live geoscience LLM.\n")
    for query, answer in zip(MINERAL_QUERIES, STATIC_ANSWERS):
        print(f"Q: {query}")
        print(f"A: {answer}\n")
        print("-" * 70)

## 6. Combined Analysis: Location Embeddings + Production-Profile Embeddings

The most powerful representation combines **two complementary views**:

| View | Embedding source | What it captures |
|------|-----------------|------------------|
| Geographic / environmental | SatCLIP (or coordinate fallback) | Terrain, climate, biome — the geotectonic setting |
| Semantic production profile | BAAI/bge-large-en-v1.5 (from notebook 04) | What and how much a country produces |

We concatenate the two embedding vectors (after L2-normalisation and optional weighting)
and re-cluster.  Countries that are **geographically similar AND have similar production
profiles** will cluster tightly — these represent true geological peer groups.

Countries that are geographically close but production-different (e.g. Norway vs. Sweden:
same biome, different mineral specialisations) will appear in different clusters, providing
richer resolution than either view alone.

**Weighting parameter `ALPHA`:**
- `ALPHA = 0.0` — purely production-profile clustering (same as notebook 04)
- `ALPHA = 1.0` — purely location-based clustering
- `ALPHA = 0.35` — default: 65% production + 35% location

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# ── Build production profiles for the geo_df countries ────────────────────────
agg_geo = agg[agg["country_iso3"].isin(geo_df["country_iso3"])].copy()

TOP_N = 5

def build_profile(group: pd.DataFrame) -> pd.Series:
    country  = group["country"].iloc[0]
    iso3     = group["country_iso3"].iloc[0]
    ranked   = group.sort_values("mean_qty", ascending=False)
    n_min    = len(ranked)
    top_str  = "; ".join(
        f"{r['commodity']}: {r['mean_qty']:,.0f}"
        for _, r in ranked.head(TOP_N).iterrows()
    )
    profile = (
        f"{country} produces {n_min} critical mineral{'s' if n_min != 1 else ''}. "
        f"Top production: {top_str}."
    )
    return pd.Series({"country": country, "iso3": iso3, "profile_text": profile})

profiles_geo = (
    agg_geo.groupby("country", group_keys=False)
    .apply(build_profile)
    .reset_index(drop=True)
)
print(f"Production profiles built: {len(profiles_geo)} countries")

# ── Encode profiles with BAAI/bge-large-en-v1.5 ───────────────────────────────
MODEL_NAME = "BAAI/bge-large-en-v1.5"
print(f"\nLoading sentence encoder: {MODEL_NAME} ...")
text_encoder = SentenceTransformer(MODEL_NAME)

INSTRUCTION = "Represent the mineral production profile of this country for clustering: "
texts = [INSTRUCTION + t for t in profiles_geo["profile_text"].tolist()]

print(f"Encoding {len(texts)} profiles ...")
text_embeddings_raw = text_encoder.encode(
    texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True
)
text_embeddings = normalize(text_embeddings_raw, norm="l2")
print(f"Text embedding shape: {text_embeddings.shape}")

In [ ]:
# ── Align indices: profiles_geo must match geo_df order ───────────────────────
combined_df = geo_df.merge(
    profiles_geo.rename(columns={"iso3": "country_iso3"}),
    on="country_iso3",
    how="inner",
    suffixes=("", "_prof"),
).reset_index(drop=True)

print(f"Countries in combined dataset: {len(combined_df)}")

# Rebuild location embeddings aligned to combined_df order
aligned_coords = combined_df[["lat", "lon"]].values.astype(np.float32)

if SATCLIP_AVAILABLE:
    coords_tensor_aligned = torch.tensor(aligned_coords)
    with torch.no_grad():
        loc_emb_aligned = satclip_model(coords_tensor_aligned).numpy()
else:
    lat_r2 = np.radians(aligned_coords[:, 0])
    lon_r2 = np.radians(aligned_coords[:, 1])
    loc_emb_aligned = np.column_stack([
        aligned_coords[:, 0], aligned_coords[:, 1],
        np.abs(aligned_coords[:, 0]),
        np.sin(lat_r2) * np.cos(lon_r2),
        np.sin(lat_r2) * np.sin(lon_r2),
        np.cos(lat_r2),
    ]).astype(np.float32)

loc_emb_aligned = normalize(loc_emb_aligned, norm="l2")

# Rebuild text embeddings aligned to combined_df
texts_aligned = [
    INSTRUCTION + row["profile_text"]
    for _, row in combined_df.iterrows()
]
text_emb_aligned_raw = text_encoder.encode(
    texts_aligned, batch_size=32, show_progress_bar=False, convert_to_numpy=True
)
text_emb_aligned = normalize(text_emb_aligned_raw, norm="l2")

# ── Weighted concatenation ─────────────────────────────────────────────────────
# ALPHA controls balance: 0.0 = text only, 1.0 = location only
ALPHA = 0.35   # 35% location weight, 65% production-profile weight

combined_emb = np.hstack([
    (1 - ALPHA) * text_emb_aligned,
    ALPHA       * loc_emb_aligned,
])
combined_emb = normalize(combined_emb, norm="l2")
print(f"Combined embedding shape: {combined_emb.shape}  (alpha={ALPHA})")

In [ ]:
# ── UMAP + HDBSCAN on combined embeddings ─────────────────────────────────────
print("Running UMAP on combined embeddings ...")
reducer_comb = umap.UMAP(
    n_components=2,
    n_neighbors=min(10, len(combined_emb) - 1),
    min_dist=0.1,
    metric="cosine",
    random_state=42,
)
umap_comb = reducer_comb.fit_transform(combined_emb)

print("Running HDBSCAN ...")
clusterer_comb = hdbscan.HDBSCAN(
    min_cluster_size=3,
    min_samples=2,
    metric="euclidean",
    cluster_selection_method="eom",
)
comb_labels = clusterer_comb.fit_predict(combined_emb)

n_comb    = len(set(comb_labels)) - (1 if -1 in comb_labels else 0)
n_noise_c = int((comb_labels == -1).sum())
print(f"Combined clusters: {n_comb}  |  Noise: {n_noise_c}")

combined_df["umap_x"]        = umap_comb[:, 0]
combined_df["umap_y"]        = umap_comb[:, 1]
combined_df["comb_cluster"]  = comb_labels
combined_df["cluster_label"] = combined_df["comb_cluster"].apply(
    lambda c: f"Cluster {c}" if c >= 0 else "Noise"
)

# ── UMAP scatter ───────────────────────────────────────────────────────────────
unique_cl = sorted(combined_df["cluster_label"].unique())
cl_only   = [l for l in unique_cl if l != "Noise"]
cmap      = {l: palette[i % len(palette)] for i, l in enumerate(cl_only)}
cmap["Noise"] = "#cccccc"

log_p2 = np.log1p(combined_df["total_production"])
combined_df["bubble_size"] = (
    (log_p2 - log_p2.min()) / (log_p2.max() - log_p2.min() + 1e-9)
    * (MAX_B - MIN_B) + MIN_B
)

fig_comb = px.scatter(
    combined_df,
    x="umap_x",
    y="umap_y",
    color="cluster_label",
    color_discrete_map=cmap,
    size="bubble_size",
    size_max=MAX_B,
    hover_name="country",
    hover_data={
        "top_mineral":      True,
        "num_minerals":     True,
        "total_production": ":.2e",
        "cluster_label":    False,
        "umap_x":           False,
        "umap_y":           False,
        "bubble_size":      False,
    },
    title=(
        f"Combined Clusters: Location Embeddings ({int(ALPHA*100)}%) "
        f"+ Production Profile ({int((1-ALPHA)*100)}%)"
    ),
    labels={"cluster_label": "Cluster", "umap_x": "UMAP-1", "umap_y": "UMAP-2"},
    template="plotly_white",
    width=950,
    height=620,
)
fig_comb.update_layout(
    legend=dict(title="Cluster", itemsizing="constant"), font_size=13
)
fig_comb.show()

In [ ]:
# ── Cluster membership summary ─────────────────────────────────────────────────
print("Combined Cluster Summary")
print("=" * 72)
for cid in sorted(set(comb_labels)):
    subset      = combined_df[combined_df["comb_cluster"] == cid]
    label       = "NOISE" if cid == -1 else f"CLUSTER {cid}"
    countries_s = ", ".join(sorted(subset["country"].tolist()))
    top_mins    = subset["top_mineral"].value_counts().head(3).index.tolist()
    avg_n       = subset["num_minerals"].mean()
    print(f"  {label}  ({len(subset)} countries)")
    print(f"  Countries     : {countries_s}")
    print(f"  Top minerals  : {', '.join(top_mins)}")
    print(f"  Avg minerals/country: {avg_n:.1f}")
    print()

## 7. Geographic Visualisation

We plot combined clusters on a world map using Plotly `scatter_geo`.
Each country centroid is represented as a circle:

- **Colour** — combined cluster identity
- **Size** — log-scaled total production
- **Hover** — country, cluster, top mineral, production total

This view lets us check whether the combined clusters make geopolitical sense:
do the Andean lithium/copper producers group together?
Do the sub-Saharan cobalt/copper producers form their own cluster?
Do the boreal Scandinavian producers separate from temperate European producers?

In [ ]:
# ── World map: clusters on geographic coordinates ──────────────────────────────
fig_map = px.scatter_geo(
    combined_df,
    lat="lat",
    lon="lon",
    color="cluster_label",
    color_discrete_map=cmap,
    size="bubble_size",
    size_max=MAX_B,
    hover_name="country",
    hover_data={
        "top_mineral":      True,
        "num_minerals":     True,
        "total_production": ":.2e",
        "cluster_label":    True,
        "lat":              False,
        "lon":              False,
        "bubble_size":      False,
    },
    projection="natural earth",
    title=(
        "Critical Mineral Producing Countries — Combined Earth FM Clusters\n"
        f"(Location {int(ALPHA*100)}% + Production profile {int((1-ALPHA)*100)}%)"
    ),
    template="plotly_white",
    width=1050,
    height=600,
)
fig_map.update_geos(
    showcoastlines=True,   coastlinecolor="#c0c0c0",
    showland=True,         landcolor="#f5f5f0",
    showocean=True,        oceancolor="#e8f4f8",
    showlakes=True,        lakecolor="#e8f4f8",
    showcountries=True,    countrycolor="#d0d0d0",
)
fig_map.update_layout(
    legend=dict(title="Cluster", itemsizing="constant"),
    font_size=12,
    margin=dict(l=0, r=0, t=60, b=0),
)
fig_map.show()

In [ ]:
# ── Choropleth: total production by country (log scale) ───────────────────────
fig_choro = px.choropleth(
    combined_df,
    locations="country_iso3",
    color=np.log1p(combined_df["total_production"]),
    hover_name="country",
    hover_data={"top_mineral": True, "num_minerals": True, "cluster_label": True},
    color_continuous_scale="YlOrRd",
    projection="natural earth",
    title="Critical Mineral Production Volume by Country (log scale, 5-year mean)",
    labels={"color": "log(1 + production)"},
    template="plotly_white",
    width=1050,
    height=550,
)
fig_choro.update_geos(
    showcoastlines=True,   coastlinecolor="#c0c0c0",
    showland=True,         landcolor="#f5f5f0",
    showocean=True,        oceancolor="#e8f4f8",
    showcountries=True,    countrycolor="#d0d0d0",
)
fig_choro.update_layout(margin=dict(l=0, r=0, t=60, b=0), font_size=12)
fig_choro.show()

## 8. Summary & Next Steps

### Earth Foundation Model Integration Status

| Model | Status in this notebook | Barrier to full integration | Recommended action |
|-------|------------------------|----------------------------|--------------------|
| **SatCLIP** | Fully integrated (or coordinate fallback) | `pip install git+.../satclip` | Run the install cell; ~200 MB checkpoint auto-downloads |
| **Prithvi-100M** | Architecture preview | Requires HLS satellite tiles | Register at NASA EarthData; download tiles for AOIs |
| **Clay** | Architecture preview | Custom checkpoint format; ~1.5 GB | `hf_hub_download('made-with-clay/Clay', ...)` |
| **K2 / GeoLLaMA** | Static fallback answers | 7 B–65 B model; needs GPU server | Deploy via vLLM on INL HPC; set `LLM_BASE_URL` |

---

### Recommended Next Steps

1. **Obtain mine-level coordinates** — Replace country centroids with deposit-level
   coordinates from USGS MRDS or S&P SNL.  This unlocks the full discriminative power
   of SatCLIP and enables per-deposit clustering.

2. **Download HLS / Sentinel-2 tiles** — For the top 20 critical mineral deposits
   worldwide, download 3–5 year time-series of multispectral imagery and run Prithvi
   or Clay to extract EO embeddings.

3. **Fine-tune on labelled deposit data** — Use [USGS MRDS deposit type labels](https://mrdata.usgs.gov/mrds/)
   to fine-tune a linear classifier head on top of the EO embeddings for deposit-type
   prediction.

4. **Deploy K2 / GeoGalactica on INL HPC** — A vLLM server with K2-7B fits on a
   single A100 80 GB GPU.  Use it to generate geological context narratives for each
   cluster, creating a fully integrated prospectivity intelligence system.

5. **Alpha sweep for combined embeddings** — Re-run the combined clustering with
   `ALPHA` values from 0.0 to 1.0 and measure cluster silhouette scores to find the
   optimal balance between geographic and production-profile information.

6. **Supply-chain risk integration** — Feed the combined cluster assignments back into
   notebook 01 (supply-chain concentration risk) to produce cluster-adjusted HHI scores:
   countries in the same geo-cluster represent correlated supply-disruption risk.

---

### Data Sources

| Source | URL | Cost |
|--------|-----|------|
| BGS World Mineral Statistics | https://www.bgs.ac.uk/mineralsuk/statistics | Free |
| NASA EarthData (HLS) | https://earthdata.nasa.gov | Free (account required) |
| Copernicus Open Access Hub | https://scihub.copernicus.eu | Free |
| USGS Earth Explorer (Landsat) | https://earthexplorer.usgs.gov | Free |
| USGS MRDS (deposit locations) | https://mrdata.usgs.gov/mrds | Free |
| SatCLIP checkpoint | https://github.com/microsoft/satclip | Free (MIT) |
| Prithvi-100M | https://huggingface.co/ibm-nasa-geospatial/Prithvi-100M | Free (Apache 2.0) |
| Clay | https://huggingface.co/made-with-clay/Clay | Free (MIT) |
| K2-7B | https://huggingface.co/daven3/k2-7b | Free |
| GeoGalactica | https://github.com/acl2023-geodiscovery/GeoGalactica | Free |